In [41]:
import time
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras import Input
from tensorflow.keras.models import Sequential

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.layers import (
    Embedding,
    Dense,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    GlobalAveragePooling1D,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
data = pd.read_csv("imdb_cleaned.csv")
print(data.shape)

(49582, 4)


In [5]:
X = data["clean_review"]
y = data["label"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print(len(X_train))
print(len(X_val))
print(len(X_test))

39665
4958
4959


In [7]:
with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

print(len(tokenizer.word_index))

90662


In [9]:
MAX_SEQUENCE_LENGTH = 500

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_integer = pad_sequences(
    X_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_integer = pad_sequences(
    X_val_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_integer = pad_sequences(
    X_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_train = np.asarray(y_train)
y_val = np.asarray(y_val)
y_test = np.asarray(y_test)

print(X_train_integer.shape)
print(X_val_integer.shape)
print(X_test_integer.shape)

(39665, 500)
(4958, 500)
(4959, 500)


In [10]:
def evaluate_model(model, X_test, y_test, model_name, batch_size=64):

    # Generate probabilities
    probabilities = model.predict(
        X_test,
        batch_size=batch_size,
        verbose=0
    ).ravel()

    # Convert probabilities to class predictions
    predictions = (probabilities >= 0.5).astype(int)

    # metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test,  predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, probabilities)

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)

    # Classification Report
    report = classification_report(
        y_test,
        predictions,
        target_names=["Negative", "Positive"],
        digits=4
    )

    # Print results
    print("=" * 60)
    print(f"{model_name} RESULTS")
    print("=" * 60)

    print(f"Accuracy : {accuracy:.5f}")
    print(f"Precision: {precision:.5f}")
    print(f"Recall   : {recall:.5f}")
    print(f"F1 Score : {f1:.5f}")
    print(f"ROC-AUC  : {roc_auc:.5f}")

    print("\nClassification Report")
    print("-" * 60)
    print(report)

    print("Confusion Matrix")
    print(cm)

    # Return everything for later comparison
    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

    return results, probabilities, predictions, cm

In [11]:
bilstm_results = []
bilstm_results.append({
    "Model": "BiLSTM Baseline",
    "Accuracy": 0.87820,
    "Precision": 0.89386,
    "Recall": 0.85938,
    "F1 Score": 0.87628,
    "ROC-AUC": 0.94213
})

In [12]:
bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.8782,0.89386,0.85938,0.87628,0.94213


**BiLSTM EarlyStopping**

In [13]:
VOCAB_SIZE = 30000
EMBEDDING_DIM = 128
MAX_SEQUENCE_LENGTH = 500

In [14]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [22]:
bilstm_early_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

bilstm_early_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_early_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history_bilstm_early = bilstm_early_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 38s 55ms/step - accuracy: 0.8108 - loss: 0.4251 - val_accuracy: 0.8788 - val_loss: 0.3067
Epoch 2/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.8893 - loss: 0.2850 - val_accuracy: 0.8868 - val_loss: 0.3123
Epoch 3/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 40s 64ms/step - accuracy: 0.9309 - loss: 0.1921 - val_accuracy: 0.8877 - val_loss: 0.2992
Epoch 4/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step - accuracy: 0.9473 - loss: 0.1498 - val_accuracy: 0.8927 - val_loss: 0.3419
Epoch 5/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 66ms/step - accuracy: 0.9584 - loss: 0.1212 - val_accuracy: 0.8752 - val_loss: 0.3748
Epoch 6/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 58ms/step - accuracy: 0.9760 - loss: 0.0755 - val_accuracy: 0.8671 - val_loss: 0.4185
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


In [24]:
bilstm_early_results, bilstm_early_probabilities, bilstm_early_predictions, bilstm_early_cm = evaluate_model(
    bilstm_early_model,
    X_test_integer,
    y_test,
    "BiLSTM EarlyStopping",
    batch_size=64
)

BiLSTM EarlyStopping RESULTS
Accuracy : 0.88203
Precision: 0.85816
Recall   : 0.91643
F1 Score : 0.88634
ROC-AUC  : 0.94557

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.9096    0.8474    0.8774      2470
    Positive     0.8582    0.9164    0.8863      2489

    accuracy                         0.8820      4959
   macro avg     0.8839    0.8819    0.8819      4959
weighted avg     0.8838    0.8820    0.8819      4959

Confusion Matrix
[[2093  377]
 [ 208 2281]]


In [25]:
bilstm_results.append(bilstm_early_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566


**BiLSTM Dropout**

In [26]:
bilstm_dropout_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

bilstm_dropout_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_dropout_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history_bilstm_dropout = bilstm_dropout_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 43s 62ms/step - accuracy: 0.7485 - loss: 0.5122 - val_accuracy: 0.8419 - val_loss: 0.3684
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 43s 67ms/step - accuracy: 0.8813 - loss: 0.3107 - val_accuracy: 0.8723 - val_loss: 0.3586
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.9263 - loss: 0.2084 - val_accuracy: 0.8868 - val_loss: 0.3408
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - accuracy: 0.9493 - loss: 0.1503 - val_accuracy: 0.8820 - val_loss: 0.3453
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 44s 71ms/step - accuracy: 0.9526 - loss: 0.1370 - val_accuracy: 0.8715 - val_loss: 0.3508
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 40s 65ms/step - accuracy: 0.9710 - loss: 0.0902 - val_accuracy: 0.8094 - val_loss: 0.8799
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 38s 61ms/step - accuracy: 0.8883 - loss: 0.2661 - val_accuracy: 0.8661 - val_loss: 0.5036
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 49s 78ms/step - accuracy: 0.9740 - loss: 0.0869 - 

In [28]:
bilstm_dropout_results, bilstm_dropout_probabilities, bilstm_dropout_predictions, bilstm_dropout_cm = evaluate_model(
    bilstm_dropout_model,
    X_test_integer,
    y_test,
    "BiLSTM Dropout",
    batch_size=64
)

BiLSTM Dropout RESULTS
Accuracy : 0.87255
Precision: 0.85780
Recall   : 0.89434
F1 Score : 0.87569
ROC-AUC  : 0.93213

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8887    0.8506    0.8693      2470
    Positive     0.8578    0.8943    0.8757      2489

    accuracy                         0.8726      4959
   macro avg     0.8733    0.8725    0.8725      4959
weighted avg     0.8732    0.8726    0.8725      4959

Confusion Matrix
[[2101  369]
 [ 263 2226]]


In [29]:
bilstm_results.append(bilstm_dropout_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129


**BiLSTM Batch Normalization**

In [32]:
bilstm_batchnorm_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])
bilstm_batchnorm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_batchnorm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,120,961 (15.72 MB)

 Trainable params: 4,120,321 (15.72 MB)

 Non-trainable params: 640 (2.50 KB)

In [33]:
history_bilstm_batchnorm = bilstm_batchnorm_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 39s 58ms/step - accuracy: 0.7871 - loss: 0.4438 - val_accuracy: 0.6987 - val_loss: 0.6491
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 43s 69ms/step - accuracy: 0.8924 - loss: 0.2630 - val_accuracy: 0.8403 - val_loss: 0.3606
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 72s 53ms/step - accuracy: 0.9382 - loss: 0.1623 - val_accuracy: 0.8076 - val_loss: 0.4657
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9601 - loss: 0.1054 - val_accuracy: 0.8697 - val_loss: 0.4195
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9733 - loss: 0.0732 - val_accuracy: 0.8526 - val_loss: 0.5479
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9803 - loss: 0.0582 - val_accuracy: 0.8233 - val_loss: 0.5958
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9817 - loss: 0.0510 - val_accuracy: 0.8562 - val_loss: 0.6510
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step - accuracy: 0.9843 - loss: 0.0451 - 

In [34]:
bilstm_batchnorm_results, bilstm_batchnorm_probabilities, bilstm_batchnorm_predictions, bilstm_batchnorm_cm = evaluate_model(
    bilstm_batchnorm_model,
    X_test_integer,
    y_test,
    "BiLSTM Batch Normalization",
    batch_size=64
)

BiLSTM Batch Normalization RESULTS
Accuracy : 0.85783
Precision: 0.86084
Recall   : 0.85496
F1 Score : 0.85789
ROC-AUC  : 0.92541

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8548    0.8607    0.8578      2470
    Positive     0.8608    0.8550    0.8579      2489

    accuracy                         0.8578      4959
   macro avg     0.8578    0.8578    0.8578      4959
weighted avg     0.8579    0.8578    0.8578      4959

Confusion Matrix
[[2126  344]
 [ 361 2128]]


In [35]:
bilstm_results.append(bilstm_batchnorm_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413


**BiLSTM Learning Rate**

In [36]:
bilstm_lr_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
bilstm_lr_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_lr_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [37]:
history_bilstm_lr = bilstm_lr_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 50ms/step - accuracy: 0.7972 - loss: 0.4285 - val_accuracy: 0.8528 - val_loss: 0.3702
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9052 - loss: 0.2514 - val_accuracy: 0.8875 - val_loss: 0.2840
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9407 - loss: 0.1686 - val_accuracy: 0.8921 - val_loss: 0.2848
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9605 - loss: 0.1170 - val_accuracy: 0.8895 - val_loss: 0.3482
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9721 - loss: 0.0855 - val_accuracy: 0.8883 - val_loss: 0.4293
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9729 - loss: 0.0796 - val_accuracy: 0.8526 - val_loss: 0.4020
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9856 - loss: 0.0469 - val_accuracy: 0.8820 - val_loss: 0.4572
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9849 - loss: 0.0459 - 

In [38]:
bilstm_lr_results, bilstm_lr_probabilities, bilstm_lr_predictions, bilstm_lr_cm = evaluate_model(
    bilstm_lr_model,
    X_test_integer,
    y_test,
    "BiLSTM Learning Rate 0.0005",
    batch_size=64
)

BiLSTM Learning Rate 0.0005 RESULTS
Accuracy : 0.87376
Precision: 0.85868
Recall   : 0.89594
F1 Score : 0.87692
ROC-AUC  : 0.93283

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8903    0.8514    0.8704      2470
    Positive     0.8587    0.8959    0.8769      2489

    accuracy                         0.8738      4959
   macro avg     0.8745    0.8737    0.8737      4959
weighted avg     0.8745    0.8738    0.8737      4959

Confusion Matrix
[[2103  367]
 [ 259 2230]]


In [39]:
bilstm_results.append(bilstm_lr_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825


**BiLSTM ReduceLR Model**

In [42]:
bilstm_reducelr_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

bilstm_reducelr_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
bilstm_reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

bilstm_reducelr_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
history_bilstm_reducelr = bilstm_reducelr_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[bilstm_reduce_lr],
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 51ms/step - accuracy: 0.7907 - loss: 0.4473 - val_accuracy: 0.8747 - val_loss: 0.3066 - learning_rate: 0.0010
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.8657 - loss: 0.3416 - val_accuracy: 0.8651 - val_loss: 0.3912 - learning_rate: 0.0010
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9223 - loss: 0.2175 - val_accuracy: 0.8998 - val_loss: 0.2801 - learning_rate: 0.0010
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 42s 51ms/step - accuracy: 0.9539 - loss: 0.1386 - val_accuracy: 0.8745 - val_loss: 0.3189 - learning_rate: 0.0010
Epoch 5/10
619/620 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9654 - loss: 0.1061
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9619 - loss: 0.1155 - val_accuracy: 0.8846 - val_loss: 0.3407 - learning_rate: 0.0010
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.9821 - los

In [44]:
bilstm_reducelr_results, bilstm_reducelr_probabilities, bilstm_reducelr_predictions, bilstm_reducelr_cm = evaluate_model(
    bilstm_reducelr_model,
    X_test_integer,
    y_test,
    "BiLSTM ReduceLR",
    batch_size=64
)

BiLSTM ReduceLR RESULTS
Accuracy : 0.88123
Precision: 0.88649
Recall   : 0.87545
F1 Score : 0.88094
ROC-AUC  : 0.93688

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8760    0.8870    0.8815      2470
    Positive     0.8865    0.8755    0.8809      2489

    accuracy                         0.8812      4959
   macro avg     0.8813    0.8812    0.8812      4959
weighted avg     0.8813    0.8812    0.8812      4959

Confusion Matrix
[[2191  279]
 [ 310 2179]]


In [45]:
bilstm_results.append(bilstm_reducelr_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825
5,BiLSTM ReduceLR,0.881226,0.886493,0.875452,0.880938,0.936884


**BiLSTM Batch Size 32**

In [46]:
bilstm_batch32_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

bilstm_batch32_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_batch32_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
history_bilstm_batch32 = bilstm_batch32_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 52s 41ms/step - accuracy: 0.7302 - loss: 0.5427 - val_accuracy: 0.5081 - val_loss: 0.6896
Epoch 2/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.8278 - loss: 0.3821 - val_accuracy: 0.8854 - val_loss: 0.2887
Epoch 3/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.9273 - loss: 0.1942 - val_accuracy: 0.8965 - val_loss: 0.2639
Epoch 4/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.9581 - loss: 0.1211 - val_accuracy: 0.8941 - val_loss: 0.3000
Epoch 5/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.9775 - loss: 0.0701 - val_accuracy: 0.8923 - val_loss: 0.3697
Epoch 6/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.9888 - loss: 0.0412 - val_accuracy: 0.8862 - val_loss: 0.3908
Epoch 7/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.9932 - loss: 0.0258 - val_accuracy: 0.8854 - val_loss: 0.4740
Epoch 8/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 0.9941 -

In [48]:
bilstm_batch32_results, bilstm_batch32_probabilities, bilstm_batch32_predictions, bilstm_batch32_cm = evaluate_model(
    bilstm_batch32_model,
    X_test_integer,
    y_test,
    "BiLSTM Batch Size 32",
    batch_size=32
)

BiLSTM Batch Size 32 RESULTS
Accuracy : 0.87760
Precision: 0.84929
Recall   : 0.91924
F1 Score : 0.88289
ROC-AUC  : 0.93390

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.9113    0.8356    0.8718      2470
    Positive     0.8493    0.9192    0.8829      2489

    accuracy                         0.8776      4959
   macro avg     0.8803    0.8774    0.8773      4959
weighted avg     0.8802    0.8776    0.8774      4959

Confusion Matrix
[[2064  406]
 [ 201 2288]]


In [49]:
bilstm_results.append(bilstm_batch32_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825
5,BiLSTM ReduceLR,0.881226,0.886493,0.875452,0.880938,0.936884
6,BiLSTM Batch Size 32,0.877596,0.849295,0.919245,0.882886,0.933898


**BiLSTM Batch Size 128**

In [50]:
bilstm_batch128_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
bilstm_batch128_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_batch128_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_8 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [51]:
history_bilstm_batch128 = bilstm_batch128_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=128,
    verbose=1
)

Epoch 1/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 26s 77ms/step - accuracy: 0.7943 - loss: 0.4392 - val_accuracy: 0.8649 - val_loss: 0.3471
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 24s 77ms/step - accuracy: 0.8925 - loss: 0.2734 - val_accuracy: 0.8695 - val_loss: 0.3382
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 24s 77ms/step - accuracy: 0.9151 - loss: 0.2277 - val_accuracy: 0.8784 - val_loss: 0.3233
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 24s 76ms/step - accuracy: 0.9481 - loss: 0.1521 - val_accuracy: 0.8715 - val_loss: 0.3269
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 24s 76ms/step - accuracy: 0.9616 - loss: 0.1141 - val_accuracy: 0.8826 - val_loss: 0.3460
Epoch 6/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 24s 76ms/step - accuracy: 0.9619 - loss: 0.1111 - val_accuracy: 0.8417 - val_loss: 0.4109
Epoch 7/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - accuracy: 0.9777 - loss: 0.0714 - val_accuracy: 0.8774 - val_loss: 0.3685
Epoch 8/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 24s 76ms/step - accuracy: 0.9841 - loss: 0.0526 - 

In [52]:
bilstm_batch128_results, bilstm_batch128_probabilities, bilstm_batch128_predictions, bilstm_batch128_cm = evaluate_model(
    bilstm_batch128_model,
    X_test_integer,
    y_test,
    "BiLSTM Batch Size 128",
    batch_size=128
)

BiLSTM Batch Size 128 RESULTS
Accuracy : 0.86550
Precision: 0.85951
Recall   : 0.87505
F1 Score : 0.86721
ROC-AUC  : 0.93002

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8718    0.8559    0.8637      2470
    Positive     0.8595    0.8751    0.8672      2489

    accuracy                         0.8655      4959
   macro avg     0.8656    0.8655    0.8655      4959
weighted avg     0.8656    0.8655    0.8655      4959

Confusion Matrix
[[2114  356]
 [ 311 2178]]


In [53]:
bilstm_results.append(bilstm_batch128_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825
5,BiLSTM ReduceLR,0.881226,0.886493,0.875452,0.880938,0.936884
6,BiLSTM Batch Size 32,0.877596,0.849295,0.919245,0.882886,0.933898
7,BiLSTM Batch Size 128,0.865497,0.859511,0.875050,0.867211,0.930020


**BiLSTM SGD**

In [54]:
bilstm_sgd_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(128)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

bilstm_sgd_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_sgd_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
history_bilstm_sgd = bilstm_sgd_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.4992 - loss: 0.6932 - val_accuracy: 0.4956 - val_loss: 0.6932
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.4982 - loss: 0.6931 - val_accuracy: 0.4948 - val_loss: 0.6931
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5024 - loss: 0.6931 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5053 - loss: 0.6931 - val_accuracy: 0.5069 - val_loss: 0.6931
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5125 - loss: 0.6930 - val_accuracy: 0.5139 - val_loss: 0.6930
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5159 - loss: 0.6930 - val_accuracy: 0.5240 - val_loss: 0.6930
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5149 - loss: 0.6930 - val_accuracy: 0.5290 - val_loss: 0.6930
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5223 - loss: 0.6929 - 

In [56]:
bilstm_sgd_results, bilstm_sgd_probabilities, bilstm_sgd_predictions, bilstm_sgd_cm = evaluate_model(
    bilstm_sgd_model,
    X_test_integer,
    y_test,
    "BiLSTM SGD",
    batch_size=64
)

BiLSTM SGD RESULTS
Accuracy : 0.52127
Precision: 0.51605
Recall   : 0.74287
F1 Score : 0.60903
ROC-AUC  : 0.53524

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5349    0.2980    0.3827      2470
    Positive     0.5160    0.7429    0.6090      2489

    accuracy                         0.5213      4959
   macro avg     0.5255    0.5204    0.4959      4959
weighted avg     0.5254    0.5213    0.4963      4959

Confusion Matrix
[[ 736 1734]
 [ 640 1849]]


In [57]:
bilstm_results.append(bilstm_sgd_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825
5,BiLSTM ReduceLR,0.881226,0.886493,0.875452,0.880938,0.936884
6,BiLSTM Batch Size 32,0.877596,0.849295,0.919245,0.882886,0.933898
7,BiLSTM Batch Size 128,0.865497,0.859511,0.875050,0.867211,0.930020
8,BiLSTM SGD,0.521274,0.516048,0.742869,0.609025,0.535240


**BiLSTM RMSprop**

In [58]:
bilstm_rmsprop_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(
        LSTM(128)
    ),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

bilstm_rmsprop_model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_rmsprop_model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ (None, 256)            │       263,168 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
history_bilstm_rmsprop = bilstm_rmsprop_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 51ms/step - accuracy: 0.6929 - loss: 0.5660 - val_accuracy: 0.8322 - val_loss: 0.4094
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.8507 - loss: 0.3734 - val_accuracy: 0.8566 - val_loss: 0.3463
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.8813 - loss: 0.3092 - val_accuracy: 0.8453 - val_loss: 0.3994
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.9020 - loss: 0.2640 - val_accuracy: 0.8893 - val_loss: 0.2853
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.9159 - loss: 0.2287 - val_accuracy: 0.8971 - val_loss: 0.2730
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 42s 50ms/step - accuracy: 0.9304 - loss: 0.1969 - val_accuracy: 0.8977 - val_loss: 0.2704
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9397 - loss: 0.1714 - val_accuracy: 0.8927 - val_loss: 0.2787
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9513 - loss: 0.1444 - 

In [60]:
bilstm_rmsprop_results, bilstm_rmsprop_probabilities, bilstm_rmsprop_predictions, bilstm_rmsprop_cm = evaluate_model(
    bilstm_rmsprop_model,
    X_test_integer,
    y_test,
    "BiLSTM RMSprop",
    batch_size=64
)

BiLSTM RMSprop RESULTS
Accuracy : 0.89635
Precision: 0.91094
Recall   : 0.87947
F1 Score : 0.89493
ROC-AUC  : 0.95005

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8826    0.9134    0.8977      2470
    Positive     0.9109    0.8795    0.8949      2489

    accuracy                         0.8964      4959
   macro avg     0.8968    0.8964    0.8963      4959
weighted avg     0.8968    0.8964    0.8963      4959

Confusion Matrix
[[2256  214]
 [ 300 2189]]


In [61]:
bilstm_results.append(bilstm_rmsprop_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825
5,BiLSTM ReduceLR,0.881226,0.886493,0.875452,0.880938,0.936884
6,BiLSTM Batch Size 32,0.877596,0.849295,0.919245,0.882886,0.933898
7,BiLSTM Batch Size 128,0.865497,0.859511,0.875050,0.867211,0.930020
8,BiLSTM SGD,0.521274,0.516048,0.742869,0.609025,0.535240
9,BiLSTM RMSprop,0.896350,0.910945,0.879470,0.894930,0.950054


**BiLSTM Dim 256**

In [62]:
bilstm_dim256_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    Bidirectional(LSTM(256)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])


bilstm_dim256_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_dim256_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_11                │ (None, 512)            │       788,480 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,661,377 (17.78 MB)

 Trainable params: 4,661,377 (17.78 MB)

 Non-trainable params: 0 (0.00 B)

In [64]:
history_bilstm_dim256 = bilstm_dim256_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 54s 87ms/step - accuracy: 0.7372 - loss: 0.5271 - val_accuracy: 0.5024 - val_loss: 0.9343
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - accuracy: 0.7338 - loss: 0.5323 - val_accuracy: 0.7586 - val_loss: 0.4911
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - accuracy: 0.6961 - loss: 0.5745 - val_accuracy: 0.8007 - val_loss: 0.4589
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - accuracy: 0.8695 - loss: 0.3122 - val_accuracy: 0.8766 - val_loss: 0.2949
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - accuracy: 0.9223 - loss: 0.2056 - val_accuracy: 0.8687 - val_loss: 0.3124
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - accuracy: 0.9482 - loss: 0.1477 - val_accuracy: 0.8721 - val_loss: 0.3533
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - accuracy: 0.9678 - loss: 0.1023 - val_accuracy: 0.8820 - val_loss: 0.3337
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 53s 86ms/step - accuracy: 0.9777 - loss: 0.0744 - 

In [65]:
bilstm_dim256_results, bilstm_dim256_probabilities, bilstm_dim256_predictions, bilstm_dim256_cm = evaluate_model(
    bilstm_dim256_model,
    X_test_integer,
    y_test,
    "BiLSTM Dim 256",
    batch_size=64
)

BiLSTM Dim 256 RESULTS
Accuracy : 0.87679
Precision: 0.86311
Recall   : 0.89675
F1 Score : 0.87961
ROC-AUC  : 0.94295

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8917    0.8567    0.8738      2470
    Positive     0.8631    0.8967    0.8796      2489

    accuracy                         0.8768      4959
   macro avg     0.8774    0.8767    0.8767      4959
weighted avg     0.8773    0.8768    0.8767      4959

Confusion Matrix
[[2116  354]
 [ 257 2232]]


In [66]:
bilstm_results.append(bilstm_dim256_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825
5,BiLSTM ReduceLR,0.881226,0.886493,0.875452,0.880938,0.936884
6,BiLSTM Batch Size 32,0.877596,0.849295,0.919245,0.882886,0.933898
7,BiLSTM Batch Size 128,0.865497,0.859511,0.875050,0.867211,0.930020
8,BiLSTM SGD,0.521274,0.516048,0.742869,0.609025,0.535240
9,BiLSTM RMSprop,0.896350,0.910945,0.879470,0.894930,0.950054


**BiLSTM Sequence Length 300**

In [67]:
MAX_SEQUENCE_LENGTH_300 = 300

X_train_seq300 = tokenizer.texts_to_sequences(X_train)
X_val_seq300 = tokenizer.texts_to_sequences(X_val)
X_test_seq300 = tokenizer.texts_to_sequences(X_test)

X_train_300 = pad_sequences(
    X_train_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_val_300 = pad_sequences(
    X_val_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_test_300 = pad_sequences(
    X_test_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

print("X_train:", X_train_300.shape)
print("X_val  :", X_val_300.shape)
print("X_test :", X_test_300.shape)

X_train: (39665, 300)
X_val  : (4958, 300)
X_test : (4959, 300)


In [68]:
bilstm_seq300_model = Sequential([
    Input(
        shape=(300,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),
    Bidirectional(LSTM(128)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

bilstm_seq300_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_seq300_model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ (None, 300, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_12                │ (None, 256)            │       263,168 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,119,681 (15.72 MB)

 Trainable params: 4,119,681 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

In [69]:
history_bilstm_seq300 = bilstm_seq300_model.fit(
    X_train_300,
    y_train,
    validation_data=(X_val_300, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 34ms/step - accuracy: 0.7491 - loss: 0.5092 - val_accuracy: 0.8415 - val_loss: 0.3731
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.8900 - loss: 0.2843 - val_accuracy: 0.8792 - val_loss: 0.3112
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.9271 - loss: 0.1994 - val_accuracy: 0.8796 - val_loss: 0.2996
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.9562 - loss: 0.1285 - val_accuracy: 0.8826 - val_loss: 0.3468
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 34ms/step - accuracy: 0.9667 - loss: 0.1008 - val_accuracy: 0.8733 - val_loss: 0.3926
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9739 - loss: 0.0791 - val_accuracy: 0.8739 - val_loss: 0.4119
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.9812 - loss: 0.0590 - val_accuracy: 0.8766 - val_loss: 0.5412
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.9771 - loss: 0.0667 - 

In [70]:
bilstm_seq300_results, bilstm_seq300_probabilities, bilstm_seq300_predictions, bilstm_seq300_cm = evaluate_model(
    bilstm_seq300_model,
    X_test_300,
    y_test,
    "BiLSTM Sequence Length 300",
    batch_size=64
)

BiLSTM Sequence Length 300 RESULTS
Accuracy : 0.86832
Precision: 0.88539
Recall   : 0.84733
F1 Score : 0.86594
ROC-AUC  : 0.92977

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8525    0.8895    0.8706      2470
    Positive     0.8854    0.8473    0.8659      2489

    accuracy                         0.8683      4959
   macro avg     0.8690    0.8684    0.8683      4959
weighted avg     0.8690    0.8683    0.8683      4959

Confusion Matrix
[[2197  273]
 [ 380 2109]]


In [71]:
bilstm_results.append(bilstm_seq300_results)

bilstm_df = pd.DataFrame(bilstm_results)
display(bilstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BiLSTM Baseline,0.878200,0.893860,0.859380,0.876280,0.942130
1,BiLSTM EarlyStopping,0.882033,0.858164,0.916432,0.886342,0.945566
2,BiLSTM Dropout,0.872555,0.857803,0.894335,0.875688,0.932129
3,BiLSTM Batch Normalization,0.857834,0.860841,0.854962,0.857892,0.925413
4,BiLSTM Learning Rate 0.0005,0.873765,0.858683,0.895942,0.876917,0.932825
5,BiLSTM ReduceLR,0.881226,0.886493,0.875452,0.880938,0.936884
6,BiLSTM Batch Size 32,0.877596,0.849295,0.919245,0.882886,0.933898
7,BiLSTM Batch Size 128,0.865497,0.859511,0.875050,0.867211,0.930020
8,BiLSTM SGD,0.521274,0.516048,0.742869,0.609025,0.535240
9,BiLSTM RMSprop,0.896350,0.910945,0.879470,0.894930,0.950054


In [72]:
bilstm_df.to_csv("bilstm_model_comparison.csv", index=False)
print("saved ")

saved 


In [73]:
bilstm_rmsprop_model.save("bilstm_rmsprop_best.keras")
print("model saved")

model saved


In [74]:
history_bilstm_rmsprop_df = pd.DataFrame(history_bilstm_rmsprop.history)
history_bilstm_rmsprop_df.to_csv("bilstm_rmsprop_training_history.csv", index=False)
print("RMSprop history saved")


RMSprop history saved
